## 4. Submit a proposal to a toy host

The host recomputes the defect from its own `targets` and ignores any
proposer-written status flag. It rejects a defect increase without changing
the current potential. Admission here means only **this local check** passed.
It says nothing about full Symbol Game certification or authorization.


In [ ]:
import sys
from pathlib import Path

_REPO = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "omega_engine/crates_rs/omega_functional").is_dir()
)
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from fractions import Fraction as Q

from omega_engine_woland.infrastructure.providers.adapters.native_functional import NativeFunctionalUnavailable

VERTICES = ("a", "b", "c")
EDGES = (("a", "b"), ("b", "c"), ("c", "a"))


def delta(potential):
    return tuple(potential[target] - potential[source] for source, target in EDGES)


def residual(targets, potential):
    return tuple(target - observed for target, observed in zip(targets, delta(potential)))


def phi1(targets, potential):
    raise NativeFunctionalUnavailable(
        "the historical fixed-section Phi1 scorer is retired; "
        "it did not compute a quotient minimum and has no native adapter"
    )


def cycle_sum(values):
    return sum(values, Q(0))


targets = (Q(1), Q(1), Q(1))
baseline = {"a": Q(0), "b": Q(0), "c": Q(0)}
print("target cycle sum:", cycle_sum(targets))
print("baseline residual:", residual(targets, baseline))
print("baseline phi1:", phi1(targets, baseline))

In [ ]:
potential = {"a": Q(0), "b": Q(1), "c": Q(2)}
print("potential coboundary:", delta(potential))
print("coboundary cycle sum:", cycle_sum(delta(potential)))
print("residual cycle sum:", cycle_sum(residual(targets, potential)))
print("candidate phi1:", phi1(targets, potential))
assert cycle_sum(delta(potential)) == Q(0)
assert cycle_sum(residual(targets, potential)) == Q(3)
assert phi1(targets, potential) >= abs(cycle_sum(targets))

In [ ]:
def toy_host_step(current, proposed, board_targets):
    if set(proposed) != set(VERTICES) or not all(
        isinstance(value, Q) for value in proposed.values()
    ):
        return dict(current), {"status": "REFUSED_CARRIER"}
    before = phi1(board_targets, current)
    after = phi1(board_targets, proposed)
    if after > before:
        return dict(current), {
            "status": "REFUSED_DEFECT_INCREASE",
            "before": before,
            "after": after,
        }
    return dict(proposed), {
        "status": "LOCAL_ZERO_DEFECT" if after == 0 else "ADMITTED_LOCAL",
        "before": before,
        "after": after,
    }


bad = {"a": Q(0), "b": Q(-1), "c": Q(-2)}
untrusted_envelope = {"potential": bad, "verified": True}
state_after_bad, bad_report = toy_host_step(baseline, untrusted_envelope["potential"], targets)
print(bad_report)
print("state retained:", state_after_bad == baseline)
assert bad_report["status"] == "REFUSED_DEFECT_INCREASE"
assert bad_report["before"] == Q(3) and bad_report["after"] == Q(5)
assert state_after_bad == baseline

In [ ]:
state_after_tie, tie_report = toy_host_step(baseline, potential, targets)
print(tie_report)
print("residual after admitted step:", residual(targets, state_after_tie))
assert tie_report["status"] == "ADMITTED_LOCAL"
assert tie_report["after"] == Q(3)

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `12_genesis_symbol_game_evidence`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
